In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv('/content/sample_data/train_final.csv')
test  = pd.read_csv('/content/sample_data/test_final.csv')
final = pd.read_csv('/content/sample_data/df_final_data.csv')

print('train:', train.shape)
print('test: ', test.shape)
print('final:', final.shape)

In [ ]:
def clean_bucket_column(df, col_name):
    df = df.copy()
    df[col_name] = df[col_name].astype(str).str.replace('>', '').str.replace('<', '')
    df[col_name] = pd.to_numeric(df[col_name], errors='coerce')
    return df

def prepare(df):
    df = clean_bucket_column(df, 'core_bucket')
    df = clean_bucket_column(df, 'memory_bucket')
    df['prev_max_to_mean_ratio'] = df['prev_max_cpu'] / df['historical_avg_cpu_mean']
    return df

train_clean = prepare(train)
test_clean  = prepare(test)

BASE_FEATURES = [
    'core_bucket', 'memory_bucket',
    'prev_avg_cpu', 'prev_max_cpu',
    'historical_avg_cpu_mean', 'historical_avg_cpu_std'
]

EXT_FEATURES = BASE_FEATURES + ['prev_max_to_mean_ratio']

imputer = SimpleImputer(strategy='median')

## 1. Linear Regression

In [ ]:
X_train = train_clean[BASE_FEATURES]
X_test  = test_clean[BASE_FEATURES]
y_train_avg = train_clean['avg_cpu_target']
y_train_max = train_clean['max_cpu_target']
y_test_avg  = test_clean['avg_cpu_target']
y_test_max  = test_clean['max_cpu_target']

X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

lr_avg = LinearRegression().fit(X_train_imp, y_train_avg)
lr_max = LinearRegression().fit(X_train_imp, y_train_max)

pred_lr_avg = lr_avg.predict(X_test_imp)
pred_lr_max = lr_max.predict(X_test_imp)

for name, y_true, y_pred in [('avg_cpu', y_test_avg, pred_lr_avg),
                               ('max_cpu', y_test_max, pred_lr_max)]:
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2)
    print(f'Linear Regression | {name}: MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}')

## 2. Ridge + Polynomial Features

In [ ]:
X_train_imp = imputer.fit_transform(train_clean[BASE_FEATURES])
X_test_imp  = imputer.transform(test_clean[BASE_FEATURES])

poly   = PolynomialFeatures(degree=2, include_bias=False)
scaler = StandardScaler()

X_train_poly = scaler.fit_transform(poly.fit_transform(X_train_imp))
X_test_poly  = scaler.transform(poly.transform(X_test_imp))

ridge_avg = Ridge(alpha=1.0).fit(X_train_poly, y_train_avg)
ridge_max = Ridge(alpha=1.0).fit(X_train_poly, y_train_max)

pred_ridge_avg = ridge_avg.predict(X_test_poly)
pred_ridge_max = ridge_max.predict(X_test_poly)

for name, y_true, y_pred in [('avg_cpu', y_test_avg, pred_ridge_avg),
                               ('max_cpu', y_test_max, pred_ridge_max)]:
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2)
    print(f'Ridge+Poly | {name}: MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}')

## 3. Random Forest

In [ ]:
X_train_imp = imputer.fit_transform(train_clean[EXT_FEATURES])
X_test_imp  = imputer.transform(test_clean[EXT_FEATURES])

rf_avg = RandomForestRegressor(
    n_estimators=200, max_depth=15,
    min_samples_split=10, min_samples_leaf=2,
    random_state=42, n_jobs=-1
).fit(X_train_imp, y_train_avg)

rf_max = RandomForestRegressor(
    n_estimators=200, max_depth=15,
    min_samples_split=10, min_samples_leaf=2,
    random_state=42, n_jobs=-1
).fit(X_train_imp, y_train_max)

pred_rf_avg = rf_avg.predict(X_test_imp)
pred_rf_max = rf_max.predict(X_test_imp)

for name, y_true, y_pred in [('avg_cpu', y_test_avg, pred_rf_avg),
                               ('max_cpu', y_test_max, pred_rf_max)]:
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2)
    print(f'Random Forest | {name}: MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}')

## 4. LightGBM

In [ ]:
X_train_imp = imputer.fit_transform(train_clean[EXT_FEATURES])
X_test_imp  = imputer.transform(test_clean[EXT_FEATURES])

lgbm_avg = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05,
    max_depth=10, random_state=42, n_jobs=-1, verbose=-1
).fit(X_train_imp, y_train_avg)

lgbm_max = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05,
    max_depth=10, random_state=42, n_jobs=-1, verbose=-1
).fit(X_train_imp, y_train_max)

pred_lgbm_avg = lgbm_avg.predict(X_test_imp)
pred_lgbm_max = lgbm_max.predict(X_test_imp)

for name, y_true, y_pred in [('avg_cpu', y_test_avg, pred_lgbm_avg),
                               ('max_cpu', y_test_max, pred_lgbm_max)]:
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2)
    print(f'LightGBM | {name}: MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}')

## 5. LightGBM + RandomizedSearchCV

In [ ]:
train_clean['prev_avg_times_std'] = train_clean['prev_avg_cpu'] * train_clean['historical_avg_cpu_std']
test_clean['prev_avg_times_std']  = test_clean['prev_avg_cpu']  * test_clean['historical_avg_cpu_std']

SEARCH_FEATURES = EXT_FEATURES + ['prev_avg_times_std']

X_train_imp = imputer.fit_transform(train_clean[SEARCH_FEATURES])
X_test_imp  = imputer.transform(test_clean[SEARCH_FEATURES])

param_dist = {
    'n_estimators':    [500, 800, 1000],
    'learning_rate':   [0.01, 0.02, 0.03],
    'num_leaves':      [15, 31, 63],
    'max_depth':       [6, 8, 10],
    'subsample':       [0.7, 0.8, 0.9],
    'colsample_bytree':[0.7, 0.8, 0.9],
    'reg_alpha':       [0, 0.1, 0.5],
    'reg_lambda':      [0, 0.1, 0.5]
}

search = RandomizedSearchCV(
    lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
    param_dist, n_iter=20, cv=3, scoring='r2',
    random_state=42, n_jobs=-1
)
search.fit(X_train_imp, y_train_max)

lgbm_opt = search.best_estimator_
pred_lgbm_opt_max = lgbm_opt.predict(X_test_imp)

mae  = mean_absolute_error(y_test_max, pred_lgbm_opt_max)
rmse = np.sqrt(mean_squared_error(y_test_max, pred_lgbm_opt_max))
r2   = 1 - np.sum((y_test_max - pred_lgbm_opt_max)**2) / np.sum((y_test_max - np.mean(y_test_max))**2)
print(f'LightGBM opt | max_cpu: MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}')
print(f'Best params: {search.best_params_}')

## 6. Ensemble RF + LightGBM

In [ ]:
X_train_imp = imputer.fit_transform(train_clean[EXT_FEATURES])
X_test_imp  = imputer.transform(test_clean[EXT_FEATURES])

rf_e = RandomForestRegressor(
    n_estimators=200, max_depth=15,
    min_samples_split=10, min_samples_leaf=2,
    random_state=42, n_jobs=-1
).fit(X_train_imp, y_train_max)

lgb_e = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05,
    max_depth=10, random_state=42, n_jobs=-1, verbose=-1
).fit(X_train_imp, y_train_max)

pred_ensemble_max = (rf_e.predict(X_test_imp) + lgb_e.predict(X_test_imp)) / 2

mae  = mean_absolute_error(y_test_max, pred_ensemble_max)
rmse = np.sqrt(mean_squared_error(y_test_max, pred_ensemble_max))
r2   = 1 - np.sum((y_test_max - pred_ensemble_max)**2) / np.sum((y_test_max - np.mean(y_test_max))**2)
print(f'Ensemble RF+LGBM | max_cpu: MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}')

## 7. LightGBM на временном ряду (деление по VM)

In [ ]:
df = final.copy()
df = clean_bucket_column(df, 'core_bucket')
df = clean_bucket_column(df, 'memory_bucket')
df = df.sort_values(['vmid', 'timestamp'])

df['avg_cpu_next'] = df.groupby('vmid')['avg_cpu_x'].shift(-1)
df['max_cpu_next'] = df.groupby('vmid')['max_cpu_x'].shift(-1)

for lag in [1, 2, 3]:
    df[f'avg_cpu_lag{lag}'] = df.groupby('vmid')['avg_cpu_x'].shift(lag)

df['avg_cpu_rolling_mean_3'] = df.groupby('vmid')['avg_cpu_x'].transform(
    lambda x: x.rolling(3, min_periods=1).mean())
df['avg_cpu_rolling_std_3'] = df.groupby('vmid')['avg_cpu_x'].transform(
    lambda x: x.rolling(3, min_periods=1).std())

df = df.dropna()

TS_FEATURES = [
    'core_bucket', 'memory_bucket',
    'avg_cpu_x', 'max_cpu_x',
    'avg_cpu_lag1', 'avg_cpu_lag2', 'avg_cpu_lag3',
    'avg_cpu_rolling_mean_3', 'avg_cpu_rolling_std_3'
]

vmids = df['vmid'].unique()
np.random.seed(42)
np.random.shuffle(vmids)
split_idx   = int(0.8 * len(vmids))
train_vmids = set(vmids[:split_idx])

train_mask = df['vmid'].isin(train_vmids)
test_mask  = ~train_mask

X_tr = df.loc[train_mask, TS_FEATURES]
X_te = df.loc[test_mask,  TS_FEATURES]
y_tr_avg = df.loc[train_mask, 'avg_cpu_next']
y_te_avg = df.loc[test_mask,  'avg_cpu_next']
y_tr_max = df.loc[train_mask, 'max_cpu_next']
y_te_max = df.loc[test_mask,  'max_cpu_next']

imp_ts = SimpleImputer(strategy='median')
X_tr_imp = imp_ts.fit_transform(X_tr)
X_te_imp = imp_ts.transform(X_te)

ts_avg = lgb.LGBMRegressor(
    n_estimators=300, learning_rate=0.05,
    max_depth=10, random_state=42, n_jobs=-1, verbose=-1
).fit(X_tr_imp, y_tr_avg)

ts_max = lgb.LGBMRegressor(
    n_estimators=300, learning_rate=0.05,
    max_depth=10, random_state=42, n_jobs=-1, verbose=-1
).fit(X_tr_imp, y_tr_max)

pred_ts_avg = ts_avg.predict(X_te_imp)
pred_ts_max = ts_max.predict(X_te_imp)

for name, y_true, y_pred in [('avg_cpu', y_te_avg, pred_ts_avg),
                               ('max_cpu', y_te_max, pred_ts_max)]:
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2)
    print(f'LightGBM TS | {name}: MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}')

## 8. ARIMA (бейзлайн, 10 VM)

In [ ]:
df_arima = final.copy()
df_arima = clean_bucket_column(df_arima, 'core_bucket')
df_arima = df_arima.sort_values(['vmid', 'timestamp'])

all_vmids = df_arima['vmid'].unique()
np.random.seed(42)
test_vmids_arima = np.random.choice(
    [v for v in all_vmids if v not in train_vmids],
    size=min(10, len(all_vmids)), replace=False
)

arima_results = []
for vmid in test_vmids_arima:
    series = df_arima[df_arima['vmid'] == vmid].sort_values('timestamp')['avg_cpu_x'].values
    if len(series) < 10:
        continue
    n = int(len(series) * 0.8)
    try:
        fit = ARIMA(series[:n], order=(5, 1, 0)).fit()
        preds = fit.forecast(steps=len(series) - n)
        arima_results.append({
            'vmid': vmid,
            'MAE':  mean_absolute_error(series[n:], preds),
            'RMSE': np.sqrt(mean_squared_error(series[n:], preds))
        })
    except Exception:
        continue

arima_df = pd.DataFrame(arima_results)
print(f'ARIMA | avg_cpu: MAE={arima_df["MAE"].mean():.4f}  RMSE={arima_df["RMSE"].mean():.4f}')

## Бизнес-метрики

**SVR** — доля интервалов, где модель недооценила нагрузку (риск нарушения SLA).  
**Savings** — отношение суммы предсказанных значений к сумме *запрошенных* ресурсов (`core_bucket`).  
**Savings_upper** — то же самое, но знаменатель — сумма *фактического максимума* (`max_cpu_target`), верхней наблюдаемой границы.

In [ ]:
requested   = test_clean['core_bucket'].values
upper_bound = y_test_max.values
y_true      = y_test_max.values

models = {
    'Linear Regression': pred_lr_max,
    'Ridge + Poly':       pred_ridge_max,
    'Random Forest':      pred_rf_max,
    'LightGBM':           pred_lgbm_max,
    'LightGBM opt':       pred_lgbm_opt_max,
    'Ensemble RF+LGBM':   pred_ensemble_max,
}

rows = []
for name, y_pred in models.items():
    y_pred = np.array(y_pred)

    svr     = np.mean(y_pred < y_true) * 100
    savings = np.sum(y_pred) / np.sum(requested)
    savings_upper = np.sum(y_pred) / np.sum(upper_bound)

    rows.append({
        'Model':          name,
        'SVR, %':         round(svr, 4),
        'Savings':        round(savings, 4),
        'Savings_upper':  round(savings_upper, 4)
    })

metrics_df = pd.DataFrame(rows)
print(metrics_df.to_string(index=False))
print('\nЦель: SVR < 0.05%  |  Savings и Savings_upper — чем меньше тем лучше')

In [ ]:
# Расчёт SLA Violation Rate для моделей
predictions = {
    'Linear Regression': pred_max_lr,
    'Ridge + Polynomial': pred_max_ridge,
    'Random Forest (100)': pred_max_rf100,
    'Random Forest (200)': pred_max_rf200,
    'LightGBM (500)': pred_max_lgb,
    'LightGBM optimized': pred_max_lgb_opt,
    'Ensemble RF+LGB': pred_ensemble,
    'LightGBM with lags (final)': pred_max
}
print("=== SLA Violation Rate (SVR) для всех моделей ===")
print(f"{'Модель':<35} {'SVR, %':<10} {'Соответствует SLA 99.95%'}")
print("-" * 70)

for name, pred in predictions.items():
    svr = np.mean(y_test_max > pred) * 100
    ok = "Да" if svr < 0.05 else "Нет"
    print(f"{name:<35} {svr:.4f}%     {ok}")

In [ ]:
# Считаем Экономию по следующей формуле: Savings = mean(y_test_max - pred) / y_test_max * 100 для процентов
# Или как mean(y_test_max - pred)для получения экономии в абсолюте
def savings(pred):
    return np.mean((y_test_max - pred)/ y_test_max * 100)
models_data = {
    'Linear Regression':     {'svr': 30.3279, 'savings': savings(pred_max_lr)},
    'Ridge + Polynomial':    {'svr': 34.5504, 'savings': savings(pred_max_ridge)},
    'Random Forest (100)':   {'svr': 36.2394, 'savings': savings(pred_max_rf100)},
    'Random Forest (200)':   {'svr': 36.2891, 'savings': savings(pred_max_rf200)},
    'LightGBM (500)':        {'svr': 37.6552, 'savings': savings(pred_max_lgb)},
    'LightGBM optimized':    {'svr': 36.6617, 'savings': savings(pred_max_lgb_opt)},
    'Ensemble RF+LGB':       {'svr': 36.3388, 'savings': savings(pred_ensemble)},
    'LightGBM + lags':       {'svr': 37.6552, 'savings': savings(pred_max)},
}

SLA_THRESHOLD = 0.05

fig, ax = plt.subplots(figsize=(10, 6))

for name, vals in models_data.items():
    color = '#E24B4A' if vals['svr'] > SLA_THRESHOLD else '#1D9E75'
    ax.scatter(vals['savings'], vals['svr'], s=100, color=color,
               edgecolors='#555', linewidths=0.8, zorder=3)
    ax.annotate(name, (vals['savings'], vals['svr']),
                textcoords='offset points', xytext=(8, 4), fontsize=8)

ax.axhline(y=SLA_THRESHOLD, color='gray', linestyle='--', linewidth=1.2,
           label='SVR = 0.05% (SLA 99.95%)')

ax.set_xlabel('Savings, % (среднее снижение аллокации)', fontsize=11)
ax.set_ylabel('SVR, % (доля нарушений SLA)', fontsize=11)
ax.set_title('SVR vs Savings — сравнение моделей', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## График SVR vs Savings

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

for ax, savings_col, title in [
    (axes[0], 'Savings',       'SVR vs Savings (от запрошенного)'),
    (axes[1], 'Savings_upper', 'SVR vs Savings_upper (от фактического максимума)')
]:
    for i, row in metrics_df.iterrows():
        ax.scatter(row[savings_col], row['SVR, %'],
                   color=colors[i], s=180, zorder=5)
        ax.annotate(row['Model'],
                    (row[savings_col], row['SVR, %']),
                    textcoords='offset points',
                    xytext=(8, 4), fontsize=9)

    ax.axhline(y=0.05, color='red', linestyle='--', linewidth=1.5,
               label='Граница SLA 99.95% (SVR = 0.05%)')
    ax.set_xlabel(savings_col, fontsize=11)
    ax.set_ylabel('SVR, %', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('svr_vs_savings.png', dpi=150)
plt.show()

Скаттерплоты


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models_preds = {
    'Linear Regression':  pred_max_lr,
    'Ridge + Poly':       pred_max_ridge,
    'Random Forest 100':  pred_max_rf100,
    'Random Forest 200':  pred_max_rf200,
    'LightGBM (500)':     pred_max_lgb,
    'LightGBM optimized': pred_max_lgb_opt,
    'Ensemble RF+LGB':    pred_ensemble,
    'LightGBM + lags':    pred_max,
}

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, (name, pred) in enumerate(models_preds.items()):
    ax = axes[i]
    ax.scatter(y_test_max, pred, alpha=0.3, s=8, color='#378ADD', label='test')

    # диагональ y = x
    lims = [0, max(y_test_max.max(), np.array(pred).max())]
    ax.plot(lims, lims, color='#E24B4A', linewidth=1.5, label='y = x')

    r2 = 1 - np.sum((y_test_max - pred)**2) / np.sum((y_test_max - np.mean(y_test_max))**2)
    mae = np.mean(np.abs(y_test_max - pred))

    ax.set_title(name, fontsize=9, fontweight='medium')
    ax.set_xlabel('Actual', fontsize=8)
    ax.set_ylabel('Predicted', fontsize=8)
    ax.text(0.05, 0.92, f'R²={r2:.3f}  MAE={mae:.1f}',
            transform=ax.transAxes, fontsize=7.5, color='gray')
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.25)

plt.suptitle('Predicted vs Actual — max_cpu_target', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

Распределения остатков

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models_preds = {
    'Linear Regression':  pred_max_lr,
    'Ridge + Poly':       pred_max_ridge,
    'Random Forest 100':  pred_max_rf100,
    'Random Forest 200':  pred_max_rf200,
    'LightGBM (500)':     pred_max_lgb,
    'LightGBM optimized': pred_max_lgb_opt,
    'Ensemble RF+LGB':    pred_ensemble,
    'LightGBM + lags':    pred_max,
}

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, (name, pred) in enumerate(models_preds.items()):
    ax = axes[i]
    residuals = np.array(pred) - np.array(y_test_max)  # pred - actual

    ax.hist(residuals, bins=40, color='#378ADD', alpha=0.6, edgecolor='#185FA5', linewidth=0.4)
    ax.axvline(0, color='#E24B4A', linewidth=1.5, label='0')
    ax.axvline(residuals.mean(), color='#1D9E75', linewidth=1.5,
               linestyle='--', label=f'mean={residuals.mean():.1f}')

    ax.set_title(name, fontsize=9, fontweight='medium')
    ax.set_xlabel('pred − actual', fontsize=8)
    ax.set_ylabel('Частота', fontsize=8)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.25)

plt.suptitle('Распределение остатков — max_cpu_target', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

Featurer importance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.impute import SimpleImputer
import lightgbm as lgb


train = pd.read_csv('/content/sample_data/train_final.csv')
test = pd.read_csv('/content/sample_data/test_final.csv')

def clean_bucket_column(df, col_name):
    df = df.copy()
    df[col_name] = df[col_name].astype(str).str.replace('>', '').str.replace('<', '')
    df[col_name] = pd.to_numeric(df[col_name], errors='coerce')
    return df

train_clean = clean_bucket_column(train, 'core_bucket')
train_clean = clean_bucket_column(train_clean, 'memory_bucket')
test_clean = clean_bucket_column(test, 'core_bucket')
test_clean = clean_bucket_column(test_clean, 'memory_bucket')

feature_columns = ['core_bucket', 'memory_bucket', 'prev_avg_cpu', 'prev_max_cpu',
                   'historical_avg_cpu_mean', 'historical_avg_cpu_std']

X_train = train_clean[feature_columns]
y_train = train_clean['max_cpu_target']
X_test = test_clean[feature_columns]
y_test = test_clean['max_cpu_target']

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

# Дополнительный признак для моделей, которые его используют
train_clean['prev_max_to_mean_ratio'] = train_clean['prev_max_cpu'] / train_clean['historical_avg_cpu_mean']
test_clean['prev_max_to_mean_ratio'] = test_clean['prev_max_cpu'] / test_clean['historical_avg_cpu_mean']
feature_columns_ratio = feature_columns + ['prev_max_to_mean_ratio']
X_train_ratio = imputer.fit_transform(train_clean[feature_columns_ratio])
X_test_ratio = imputer.transform(test_clean[feature_columns_ratio])

# 2. Обучение моделей и получение важности/коэффициентов
# 2.1 Linear Regression
lr = LinearRegression()
lr.fit(X_train_imp, y_train)
lr_coef = np.abs(lr.coef_)

# 2.2 Ridge + Polynomial (degree=2)
poly = PolynomialFeatures(degree=2, include_bias=False)
scaler = StandardScaler()
X_train_poly = scaler.fit_transform(poly.fit_transform(X_train_imp))
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_poly, y_train)
ridge_coef = np.abs(ridge.coef_)
poly_feature_names = poly.get_feature_names_out(feature_columns)

# 2.3 Random Forest (100)
rf100 = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf100.fit(X_train_imp, y_train)
rf100_imp = rf100.feature_importances_

# 2.4 Random Forest (200) с ratio
rf200 = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_split=10, min_samples_leaf=2, random_state=42, n_jobs=-1)
rf200.fit(X_train_ratio, y_train)
rf200_imp = rf200.feature_importances_

# 2.5 LightGBM (500)
lgb500 = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=10, random_state=42, n_jobs=-1, verbose=-1)
lgb500.fit(X_train_ratio, y_train)
lgb500_imp = lgb500.feature_importances_

# 2.6 LightGBM optimized
lgb_opt = lgb.LGBMRegressor(
    subsample=0.8, reg_lambda=0.5, reg_alpha=0, num_leaves=63,
    n_estimators=1000, max_depth=8, learning_rate=0.01,
    colsample_bytree=0.7, random_state=42, n_jobs=-1, verbose=-1
)
lgb_opt.fit(X_train_ratio, y_train)
lgb_opt_imp = lgb_opt.feature_importances_

# 2.7 Ensemble (усреднение RF200 + LightGBM500) – importance не определена, пропустим
def plot_importance(importances, feature_names, title, color='steelblue', top_n=None):
    if top_n is not None:
        idx = np.argsort(importances)[::-1][:top_n]
    else:
        idx = np.argsort(importances)[::-1]
    names = [feature_names[i] for i in idx]
    vals = importances[idx]
    plt.figure(figsize=(10, 6))
    plt.barh(names, vals, color=color)
    plt.xlabel('Importance')
    plt.title(title)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

# 4. Построение графиков для каждой модели
plot_importance(lr_coef, feature_columns, 'Linear Regression - абсолютные коэффициенты', color='skyblue')
plot_importance(ridge_coef, poly_feature_names, 'Ridge + Polynomial - абсолютные коэффициенты', color='lightgreen')
plot_importance(rf100_imp, feature_columns, 'Random Forest (100) - Gini importance', color='orange')
plot_importance(rf200_imp, feature_columns_ratio, 'Random Forest (200) - Gini importance', color='coral')
plot_importance(lgb500_imp, feature_columns_ratio, 'LightGBM (500) - Gain importance', color='purple')
plot_importance(lgb_opt_imp, feature_columns_ratio, 'LightGBM optimized - Gain importance', color='magenta')